## System Architecture

- **AMAN**: arrival sequencing
- **DMAN**: departure sequencing + ATFM constraints
- **GENERATOR**: adversarial scenario mutation
- **SUPERVISOR**: rotating preference profile

Training signal is role-specific reward shaping with cross-role conflict penalties and supervisor alignment.

In [ ]:
import os
import sys
from pathlib import Path

# Offline + deterministic constraints
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('REWARD_FAILURE_MODE', 'strict')

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workspace root:', ROOT)
print('TRANSFORMERS_OFFLINE=', os.getenv('TRANSFORMERS_OFFLINE'))
print('HF_HUB_OFFLINE=', os.getenv('HF_HUB_OFFLINE'))

In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Model Loading (offline, cache-only)

We resolve model files from local cache only and avoid any live Hub lookup.

In [ ]:
MODEL_REF = 'Qwen/Qwen2.5-7B-Instruct'  # or local path

def resolve_model_path(model_ref: str) -> str:
    p = Path(model_ref)
    if p.exists():
        return str(p)
    return snapshot_download(repo_id=model_ref, local_files_only=True, resume_download=True)

model_path = resolve_model_path(MODEL_REF)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)

print('Resolved model path:', model_path)
print('dtype:', dtype)

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    torch_dtype=dtype,
    device_map='auto' if torch.cuda.is_available() else None,
)
print('Model loaded.')

## LoRA Setup

LoRA is used to keep optimization lightweight and stable on constrained server setup.
We avoid full-model finetuning for memory and recovery safety.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print('LoRA attached and optimizer created.')

## Dataset / Episode Simulation

Each sample is one role-turn containing chat prompt + role metadata.

In [ ]:
from training.dataset import build_episode_dataset

samples = build_episode_dataset(n_episodes=5, seed=42, include_generator=True, include_supervisor=True)
print('Total samples:', len(samples))
print('Sample keys:', list(samples[0].keys()))
print('Sample role:', samples[0]['agent_role'])

## Reward Function Overview

Reward logic is reused from repository code (unchanged scoring behavior):
- `aman_reward_fn`: delay, emergency handling, coverage, ToM bonus, supervisor alignment, normalized conflict penalty
- `dman_reward_fn`: delay, ATFM compliance, emergency handling, coverage, ToM bonus, supervisor alignment, normalized conflict penalty
- `generator_reward_fn`: adversarial reward with solvability guard
- `supervisor_reward_fn`: preference-alignment scoring and calibration

This notebook does not alter those formulas.

In [ ]:
from training.reward_functions import aman_reward_fn, dman_reward_fn, generator_reward_fn, supervisor_reward_fn

ROLE_TO_REWARD_FN = {
    'AMAN': aman_reward_fn,
    'DMAN': dman_reward_fn,
    'GENERATOR': generator_reward_fn,
    'SUPERVISOR': supervisor_reward_fn,
}

def compute_reward(sample, completion):
    role = sample['agent_role']
    fn = ROLE_TO_REWARD_FN[role]

    if role == 'AMAN':
        return fn([completion],
                  task_id=[sample['task_id']],
                  supervisor_profile=[sample['supervisor_profile']],
                  dman_slots_json=[sample.get('dman_slots_json', '[]')],
                  atfm_deadlines_json=[sample.get('atfm_deadlines_json', '{}')])[0]

    if role == 'DMAN':
        return fn([completion],
                  task_id=[sample['task_id']],
                  supervisor_profile=[sample['supervisor_profile']],
                  aman_slots_json=[sample.get('aman_slots_json', '[]')],
                  atfm_deadlines_json=[sample.get('atfm_deadlines_json', '{}')])[0]

    if role == 'GENERATOR':
        return fn([completion],
                  task_id=[sample['task_id']],
                  controller_scores=[float(sample.get('controller_scores', 0.5))])[0]

    return fn([completion],
              task_id=[sample['task_id']],
              supervisor_profile=[sample['supervisor_profile']],
              merged_plan_json=[sample.get('merged_plan_json', '[]')])[0]

## Training Loop (Harness)

This is a controlled offline harness loop for runtime validation.
It logs role, reward, and loss per step.

In [ ]:
MAX_STEPS = min(20, len(samples))
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 128

rewards = []
losses = []
roles = []

model.train()
for step in range(MAX_STEPS):
    sample = samples[step]
    prompt_text = tokenizer.apply_chat_template(sample['prompt'], tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reward = float(compute_reward(sample, completion))
    reward = max(-1.0, min(1.0, reward))

    model.train()
    outputs = model(**inputs, labels=inputs['input_ids'])
    ce_loss = outputs.loss

    weight = max(0.1, 1.0 - reward)
    final_loss = ce_loss * weight

    if torch.isnan(final_loss) or torch.isinf(final_loss):
        print(f'[WARN] step={step} unstable loss; skipped')
        optimizer.zero_grad(set_to_none=True)
        continue

    final_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    role = sample['agent_role']
    rewards.append(reward)
    losses.append(float(ce_loss.item()))
    roles.append(role)

    print(f'step={step:03d} role={role:<10} reward={reward:+.4f} ce_loss={ce_loss.item():.4f}')

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(rewards)
plt.title('Reward over Steps')
plt.xlabel('Step')
plt.ylabel('Reward')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title('Cross-Entropy Loss over Steps')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
df = pd.DataFrame({'role': roles, 'reward': rewards})
plt.figure(figsize=(10, 5))
df.groupby('role')['reward'].mean().sort_values().plot(kind='bar')
plt.title('Average Reward per Role')
plt.xlabel('Role')
plt.ylabel('Average Reward')
plt.grid(True, axis='y')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(rewards, losses, alpha=0.8)
plt.title('Reward vs Loss Relationship')
plt.xlabel('Reward')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## Observations

Fill this section after execution with concrete outcomes, e.g.:
- supervisor role reward trend
- generator penalty behavior
- AMAN/DMAN stability or oscillation
- whether loss decreases with bounded rewards

## Limitations

- This harness is not full GRPO trajectory/group optimization.
- Reward scaling here is a stability proxy, not final policy objective.
- No multi-sample policy grouping or KL-controlled GRPO updates in this notebook run.

## Next Steps

1. Keep this harness as server-stability gate.
2. Move to full GRPO runner after environment lock is proven.
3. Add per-role moving averages and longer-run diagnostics once stable.